# 04 - Causal Forest and final evaluation

`econml.grf.CausalForest` -- an honest, tree-ensemble CATE estimator -- and
then the final comparison of every model on the untouched test partition.

Run notebooks `01`, `02` and `03` first: the final comparison reads the test
predictions each of them saved.

In [ ]:
import sys
from pathlib import Path


def _find_repo_root() -> Path:
    """Locate the repo root without assuming the working directory.

    A fresh Kaggle kernel starts in /kaggle/working, not in the repo, so we
    search: the current directory and its parents (local development), then
    /kaggle/working and each attached /kaggle/input/<slug>/ (Kaggle, where
    the repo is cloned into working or attached as a dataset).
    """
    bases = [Path.cwd(), *Path.cwd().parents, Path("/kaggle/working"), Path("/kaggle/input")]
    for base in bases:
        if not base.is_dir():
            continue
        if (base / "src" / "data.py").is_file():
            return base
        for child in sorted(p for p in base.iterdir() if p.is_dir()):
            if (child / "src" / "data.py").is_file():
                return child
    raise RuntimeError(
        "Could not locate the repository root (no src/data.py found). On Kaggle, "
        "clone this repository into /kaggle/working or attach it as a dataset."
    )


REPO_ROOT = _find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print("repo root:", REPO_ROOT)

In [ ]:
# Kaggle's stock image does not include econml (and may not match the
# pinned lightgbm version). Installing quietly here is a no-op if the pinned
# versions are already present -- e.g. local dev via requirements.txt.
%pip install -q econml==0.17.0 lightgbm==4.7.0
import econml
import lightgbm
print("econml:", econml.__version__, " lightgbm:", lightgbm.__version__)

In [ ]:
from src.data import PRIMARY_OUTCOME, TREATMENT_COLUMN, load_config, load_parquet, output_dir, save_parquet
from src.preprocessing import CausalForestCategoricalEncoder
from src.models import fit_causal_forest, predict_causal_forest_tau
from src.evaluation import evaluate_ranking

CONFIG = load_config()
OUTPUT_DIR = output_dir()

train_frame = load_parquet(OUTPUT_DIR / "train.parquet")
val_frame = load_parquet(OUTPUT_DIR / "validation.parquet")
test_frame = load_parquet(OUTPUT_DIR / "test.parquet")
Y_train, Y_val, Y_test = (f[PRIMARY_OUTCOME] for f in (train_frame, val_frame, test_frame))
T_train, T_val, T_test = (f[TREATMENT_COLUMN] for f in (train_frame, val_frame, test_frame))
train_frame.shape

## Categorical representation

`CausalForest` consumes a dense numeric matrix and has no LightGBM-equivalent
native categorical support. Passing the raw categorical tokens through as
floats would fabricate an ordering the forest's splits would then exploit --
the same error the feature-semantics decision exists to prevent, which is
why `fit_causal_forest` rejects raw categorical columns outright.

The representation used is **frequency-capped top-K one-hot**: keep the top
`K` categories by train frequency, bucket the rest (including categories
unseen at fit time) into an explicit `OTHER` level, one-hot encode, and pass
the continuous features through unchanged. `K` is chosen for resource
feasibility, never by comparing model performance across `K`.

In [ ]:
encoder = CausalForestCategoricalEncoder(k=CONFIG["causal_forest"]["categorical_top_k"]).fit(train_frame)
X_train_cf, X_val_cf, X_test_cf = (encoder.transform(f) for f in (train_frame, val_frame, test_frame))
print("encoded feature count:", X_train_cf.shape[1])
X_train_cf.shape, X_val_cf.shape, X_test_cf.shape

## Fit

`honest=True` and `n_jobs=1` are load-bearing, not defaults left untouched:
honesty is what makes the leaf estimates valid, and `n_jobs=1` is required
for reproducibility (`n_jobs=2`/`-1` were found to produce different
predictions than `n_jobs=1` under an identical `random_state`).

This is the most expensive fit in the project -- on full CRITEO it dominates
the Kaggle runtime budget.

In [ ]:
causal_forest = fit_causal_forest(X_train_cf, T_train, Y_train, seed=CONFIG["seed"])
print("fitted:", type(causal_forest).__name__)

In [ ]:
cf_val_scores = predict_causal_forest_tau(causal_forest, X_val_cf)
cf_val_ranking = evaluate_ranking(cf_val_scores, T_val, Y_val)
print("Causal Forest validation qini_above_random:", round(cf_val_ranking.qini_above_random, 4))
print("Causal Forest validation auuc_above_random:", round(cf_val_ranking.auuc_above_random, 4))
cf_val_ranking.uplift_at_k

In [ ]:
import pandas as pd

cf_test_predictions = pd.DataFrame({
    "score": predict_causal_forest_tau(causal_forest, X_test_cf),
    TREATMENT_COLUMN: T_test.to_numpy(),
    PRIMARY_OUTCOME: Y_test.to_numpy(),
})
save_parquet(cf_test_predictions, OUTPUT_DIR / "preds_causal_forest_test.parquet")
print("saved ->", OUTPUT_DIR / "preds_causal_forest_test.parquet")

# Final evaluation

Everything below is scored on the **test** partition, which no model was
fit on, early-stopped against, or selected with. This is the first and only
place it is used.

A theoretical random-ranking reference is included as the honest floor: an
uplift model that cannot beat it has not earned its complexity.

In [ ]:
import numpy as np
import pandas as pd

MODEL_FILES = {
    "Response LightGBM": "preds_response_test.parquet",
    "T-Learner": "preds_tlearner_test.parquet",
    "X-Learner": "preds_xlearner_test.parquet",
    "Causal Forest": "preds_causal_forest_test.parquet",
}

predictions, rankings = {}, {}
for label, filename in MODEL_FILES.items():
    path = OUTPUT_DIR / filename
    if not path.exists():
        print(f"skipping {label}: {filename} not found -- run its notebook first")
        continue
    frame = load_parquet(path)
    predictions[label] = frame
    rankings[label] = evaluate_ranking(frame["score"], frame[TREATMENT_COLUMN], frame[PRIMARY_OUTCOME])

# Theoretical random reference, on the same test rows.
reference = next(iter(predictions.values()))
rng = np.random.default_rng(CONFIG["seed"])
rankings["Random (reference)"] = evaluate_ranking(
    rng.uniform(size=len(reference)), reference[TREATMENT_COLUMN], reference[PRIMARY_OUTCOME]
)
sorted(rankings)

## Comparison table

In [ ]:
rows = []
for label, r in rankings.items():
    row = {
        "model": label,
        "qini_above_random": r.qini_above_random,
        "auuc_above_random": r.auuc_above_random,
        "qini_area": r.qini_area,
        "auuc_area": r.auuc_area,
    }
    row.update({f"uplift@{k}": v for k, v in r.uplift_at_k.items()})
    rows.append(row)

comparison = pd.DataFrame(rows).set_index("model").sort_values("qini_above_random", ascending=False)
comparison.round(5)

In [ ]:
save_parquet(comparison.reset_index(), OUTPUT_DIR / "final_comparison.parquet")
comparison[["qini_above_random", "auuc_above_random"]].plot.barh(
    figsize=(9, 4), title="Test-set ranking performance above the random reference"
)

## Qini and uplift curves, all models

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
for label, r in rankings.items():
    style = "--" if label.startswith("Random") else "-"
    ax1.plot(r.qini_curve["coverage"], r.qini_curve["qini_gain"], style, label=label)
    ax2.plot(r.uplift_curve["coverage"], r.uplift_curve["uplift_gain"], style, label=label)
ax1.set_xlabel("Coverage"); ax1.set_ylabel("Qini gain"); ax1.set_title("Qini curves (test)"); ax1.legend()
ax2.set_xlabel("Coverage"); ax2.set_ylabel("Uplift gain"); ax2.set_title("Uplift curves / AUUC (test)"); ax2.legend()
fig.tight_layout()

## CATE analysis

The metrics above say which ranking is *better*; these say what the models
actually believe. Three things worth checking:

1. **Distribution** -- a model whose predicted CATE is nearly constant is not
   finding heterogeneity, whatever its Qini says.
2. **Decile monotonicity** -- if the ranking is real, observed uplift should
   decline from the top decile down.
3. **Agreement** -- where the causal estimators agree, the signal is more
   plausibly real than where only one of them sees it.

In [ ]:
causal_models = [m for m in ("T-Learner", "X-Learner", "Causal Forest") if m in predictions]

cate_summary = pd.DataFrame({
    label: pd.Series(predictions[label]["score"]).describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95])
    for label in causal_models
}).T
cate_summary.round(6)

In [ ]:
fig, axes = plt.subplots(1, max(len(causal_models), 1), figsize=(5 * max(len(causal_models), 1), 3.5), squeeze=False)
for ax, label in zip(axes[0], causal_models):
    ax.hist(predictions[label]["score"], bins=60)
    ax.axvline(0.0, color="k", linestyle="--", linewidth=1)
    ax.set_title(f"{label}\npredicted CATE")
fig.tight_layout()

In [ ]:
# Observed uplift by predicted-CATE decile: decile 1 = highest predicted uplift.
decile_view = pd.DataFrame({
    label: rankings[label].decile_table.set_index("decile")["observed_uplift"] for label in causal_models
})
decile_view.round(5)

In [ ]:
ax = decile_view.plot(marker="o", figsize=(9, 4),
                      title="Observed test uplift by predicted-CATE decile (1 = highest predicted)")
ax.axhline(0.0, color="k", linestyle="--", linewidth=1)
ax.set_xlabel("Decile of predicted CATE"); ax.set_ylabel("Observed uplift")

In [ ]:
if len(causal_models) > 1:
    agreement = pd.DataFrame({label: predictions[label]["score"].to_numpy() for label in causal_models}).corr(
        method="spearman"
    ).round(3)
    print("Spearman rank correlation between causal models' predicted CATE:")
    print(agreement)
else:
    print("need at least two causal models for an agreement matrix")

## Conclusions

Read the comparison table and the decile plot together, then state the
result in the form the evidence actually supports:

> Among the evaluated implementations, **<model>** achieved the strongest
> test-set uplift ranking on CRITEO-UPLIFTv2.1 under this protocol
> (Qini above random = *<value>*, AUUC above random = *<value>*).

Things this evidence does **not** establish, and that a careful reader will
check you haven't claimed:

- that the winning estimator is universally best -- this is one dataset, one
  implementation of each method, one hyperparameter setting, one metric family;
- that a meta-learner family is intrinsically superior -- what is measured is
  a *specific implementation* on *these* features;
- that predicted CATE is a true individual treatment effect -- both potential
  outcomes are never observed for anyone, which is also why no PEHE against
  ground truth is reported here;
- that the response model's ranking is causal, however strong its AUC is.

The most informative comparison in the table is usually Response LightGBM
versus the causal estimators: it separates "who converts" from "who converts
*because of* the treatment," which is the entire premise of uplift modeling.